# LeetCode 823 — Binary Trees With Factors bằng Python

Link đề: https://leetcode.com/problems/binary-trees-with-factors/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Dynamic Programming (quy hoạch động), Sorting

Skeleton LeetCode:

```python
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        ...
```

Ghi chú: đếm **số cây nhị phân** mà mỗi nút trong là **tích hai nút con**, mọi giá trị lấy từ `arr` (dùng lại tùy ý). Ý tưởng cốt lõi: số cây có gốc `v` là **`1` (cây một nút) cộng tổng các tích `f(a) * f(b)`** trên các cặp có `a * b = v`. Vì `a, b < v` nên sắp xếp tăng dần rồi tính `f` từ nhỏ tới lớn. Kết quả lấy theo modulo `10^9 + 7`; notebook tự kiểm chứng phép modulo bằng **số nguyên lớn chính xác** và bằng **oracle liệt kê từng cây**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi nút lá hoặc nút trong lấy giá trị từ `arr`; **mỗi nút trong bằng tích giá trị hai con**; đếm mọi cây nhị phân (có phân biệt con trái/phải) có thể tạo, theo modulo `10^9 + 7`.
2. Nắm các công cụ Python cần thiết: `dict` tra chỉ số theo giá trị, `sorted`, phép chia hết (`%`, `//`), `math.isqrt`, số học modulo, và `functools.cache` cho đệ quy có nhớ.
3. Hiểu **nguyên lý nhân** (các cây con độc lập nên nhân số cây) và **tính thứ tự** (con trái khác con phải: cặp `(a, b)` với `a != b` cho hai cây).
4. Đi qua 5 cách: định nghĩa đệ quy không nhớ, đệ quy có nhớ, DP sắp xếp (kéo), DP đẩy theo cặp, DP đối xứng chỉ duyệt thừa số `<= sqrt`; đối chiếu bằng oracle **liệt kê từng cây** và bằng số nguyên lớn không modulo.
5. Tránh các lỗi kinh điển: quên sắp xếp, quên cây một nút, chỉ đếm một thứ tự con, chia không kiểm tra chia hết, quên modulo (hoặc lấy modulo sai chỗ).

## 2. Đề bài & ví dụ

Cho mảng các số nguyên **phân biệt** `arr`, mỗi `arr[i]` **lớn hơn 1**.

Ta dựng các cây nhị phân từ các số này, **mỗi số được dùng bao nhiêu lần cũng được**. Giá trị của mỗi **nút trong** (không phải lá) phải **bằng tích giá trị hai nút con**.

Trả về **số cây nhị phân** có thể tạo, theo **modulo** `10^9 + 7`.

**Ví dụ 1**

- Input: `arr = [2, 4]`
- Output: `3`
- Giải thích: các cây là `[2]`, `[4]` và `[4, 2, 2]` (gốc `4` có hai con `2`, `2`).

**Ví dụ 2**

- Input: `arr = [2, 4, 5, 10]`
- Output: `7`
- Giải thích: `[2]`, `[4]`, `[5]`, `[10]`, `[4, 2, 2]`, `[10, 2, 5]`, `[10, 5, 2]`.

**Ràng buộc:**

- `1 <= arr.length <= 1000`
- `2 <= arr[i] <= 10^9`
- Mọi giá trị trong `arr` **khác nhau**.

Chú ý: `[10, 2, 5]` và `[10, 5, 2]` là **hai cây khác nhau** (con trái và con phải được phân biệt).

In [1]:
from typing import List
from functools import cache
from math import isqrt
import random, time
from itertools import product

MOD = 10 ** 9 + 7

examples = [
    ([2, 4], 3),
    ([2, 4, 5, 10], 7),
]
for arr, expected in examples:
    print(f"arr={arr} kỳ vọng={expected}")

arr=[2, 4] kỳ vọng=3
arr=[2, 4, 5, 10] kỳ vọng=7


## 3. Kiến thức nền

### 3.1. Cây nhị phân và nguyên lý nhân

Một cây gốc `v` là **một nút duy nhất** `v`, hoặc **gốc `v` với hai cây con** gốc `a` (trái) và `b` (phải) mà `a * b = v`. Hai cây con được chọn **độc lập**: số cách chọn cặp (cây trái, cây phải) là **tích** `f(a) * f(b)` (nguyên lý nhân). Vì con trái/phải phân biệt nên cặp có thứ tự `(a, b)` và `(b, a)` cho các cây **khác nhau** (khi `a != b`).

In [2]:
# Ví dụ nhỏ: các cây gốc 10 trong arr = [2, 4, 5, 10] (viết thủ công)
trees_of_10 = ["10", "10(2,5)", "10(5,2)"]
print(len(trees_of_10), "cây gốc 10:", trees_of_10)
print("số cây gốc 4: 2  ->", ["4", "4(2,2)"])
print("tổng: 1 (cây gốc 2) + 2 + 1 (cây gốc 5) + 3 =", 1 + 2 + 1 + 3)

3 cây gốc 10: ['10', '10(2,5)', '10(5,2)']
số cây gốc 4: 2  -> ['4', '4(2,2)']
tổng: 1 (cây gốc 2) + 2 + 1 (cây gốc 5) + 3 = 7


### 3.2. `dict` để tra theo giá trị

Ta cần hỏi "`c = v // a` có trong `arr` không?" trong `O(1)`: dùng `dict` `giá trị -> chỉ số` (hoặc `set`). `in` trên `dict` kiểm tra khóa.

In [3]:
arr = [2, 4, 5, 10]
idx = {v: i for i, v in enumerate(arr)}
print(idx, 4 in idx, 3 in idx)
v = 10
print([(a, v // a) for a in arr if v % a == 0 and v // a in idx])

{2: 0, 4: 1, 5: 2, 10: 3} True False
[(2, 5), (5, 2)]


### 3.3. Chia hết và chia nguyên

`v % a == 0` kiểm tra `a` là ước của `v`. Khi chia hết, `v // a` là thương **nguyên chính xác**. Tuyệt đối không dùng `/` (ra số thực, có thể sai số với số lớn).

In [4]:
v = 10 ** 9
for a in (2, 3, 8):
    print(a, v % a == 0, v // a, "| float:", v / a)
big = 999999999999999989
print("v / a với số lớn mất chính xác:", int(big / 1) == big, big / 1)

2 True 500000000 | float: 500000000.0
3 False 333333333 | float: 333333333.3333333
8 True 125000000 | float: 125000000.0
v / a với số lớn mất chính xác: False 1e+18


### 3.4. Số học modulo

Kết quả có thể rất lớn nên lấy `% MOD` với `MOD = 10^9 + 7`. Hai quy tắc:

- `(a + b) % m = ((a % m) + (b % m)) % m`
- `(a * b) % m = ((a % m) * (b % m)) % m`

Python có số nguyên lớn nên **không tràn** (khác Java/C++), nhưng vẫn phải `% MOD` ở cuối (và nên lấy `% MOD` trong vòng lặp để số không phình ra). Ta có thể kiểm tra lấy modulo "giữa chừng" cho kết quả đúng bằng cách so với phép tính **không modulo** rồi lấy `%` ở cuối.

In [5]:
a, b, c = 10 ** 18 + 3, 10 ** 17 + 7, 12345678901234567
direct = (a * b + c) % MOD
stepwise = ((a % MOD) * (b % MOD) + c % MOD) % MOD
print(direct, stepwise, direct == stepwise)

414815437 414815437 True


### 3.5. `math.isqrt` và đối xứng thừa số

`isqrt(v)` là căn bậc hai **nguyên** (làm tròn xuống), không sai số. Với `v = a * b`, nếu `a <= b` thì `a <= isqrt(v)`: chỉ cần duyệt thừa số **nhỏ** tới `isqrt(v)` rồi nhân đôi vì thứ tự (con trái/phải). Dùng ở Cách 5.

In [6]:
for v in (10, 16, 36, 10 ** 9):
    print(v, "isqrt =", isqrt(v), "| isqrt^2 <= v < (isqrt+1)^2:", isqrt(v) ** 2 <= v < (isqrt(v) + 1) ** 2)

10 isqrt = 3 | isqrt^2 <= v < (isqrt+1)^2: True
16 isqrt = 4 | isqrt^2 <= v < (isqrt+1)^2: True
36 isqrt = 6 | isqrt^2 <= v < (isqrt+1)^2: True
1000000000 isqrt = 31622 | isqrt^2 <= v < (isqrt+1)^2: True


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Gọi `f(v)` là số cây có **gốc `v`**. Đáp án là `Σ f(v)` trên mọi `v` trong `arr`.

`f(v) = 1 + Σ (a, b) có thứ tự, a * b = v, a, b ∈ arr:  f(a) * f(b)`

Số `1` là cây chỉ gồm nút `v`.

Với `arr = [2, 4, 5, 10]`:

| `v` | các cặp `(a, b)` có `a * b = v` trong `arr` | `f(v)` |
|---|---|---|
| 2 | không (nếu có thì `a, b < 2`, mà `arr` không có số nào nhỏ hơn 2 ngoài 2 và 2 * 1 không hợp lệ) | `1` |
| 4 | `(2, 2)` | `1 + f(2) * f(2) = 2` |
| 5 | không | `1` |
| 10 | `(2, 5)` và `(5, 2)` | `1 + f(2) f(5) + f(5) f(2) = 3` |

Tổng `1 + 2 + 1 + 3 = 7`. Thứ tự tính: **tăng dần**, vì `a, b < v` (mọi phần tử `> 1`).

In [7]:
def f_table(arr):
    arr = sorted(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = {}
    for v in arr:
        total = 1
        for a in arr:
            if a < v and v % a == 0 and v // a in idx:
                total += f[a] * f[v // a]
        f[v] = total
    return f

f = f_table([2, 4, 5, 10])
print(f, "-> tổng =", sum(f.values()))
assert sum(f.values()) == 7

{2: 1, 4: 2, 5: 1, 10: 3} -> tổng = 7


## 5. Cách 1 — Brute Force: đệ quy theo định nghĩa (không nhớ)

**Ý tưởng.** Cài đặt `f(v)` đúng như công thức mục 4 bằng đệ quy **không ghi nhớ**: duyệt mọi `a` trong `arr` và gọi `f(a)`, `f(v // a)` mỗi lần.

**Oracle độc lập.** Công thức `f` có thể sai ở chỗ nào đó; ta dựng thêm một oracle **liệt kê từng cây** (dưới dạng tuple lồng nhau) rồi đếm `len`. Chỉ dùng cho mảng nhỏ.

In [8]:
def count_rec(arr: List[int]) -> int:
    values = set(arr)

    def f(v: int) -> int:
        total = 1
        for a in arr:
            if v % a == 0 and v // a in values and a < v and v // a < v:
                total += f(a) * f(v // a)
        return total

    return sum(f(v) for v in arr) % MOD

def enumerate_trees(arr: List[int], v: int):
    """Mọi cây gốc v (dạng tuple lồng nhau): nút lá = (v,), nút trong = (v, cây_trái, cây_phải)."""
    values = set(arr)
    trees = [(v,)]
    for a in arr:
        if v % a == 0 and v // a in values:
            b = v // a
            if a < v and b < v:
                for left in enumerate_trees(arr, a):
                    for right in enumerate_trees(arr, b):
                        trees.append((v, left, right))
    return trees

def oracle(arr: List[int]) -> int:
    return sum(len(enumerate_trees(arr, v)) for v in arr) % MOD

for arr, expected in examples:
    assert count_rec(arr) == expected == oracle(arr)
print("đệ quy thuần OK; oracle liệt kê cây khớp")
print("các cây của [2, 4, 5, 10]:")
for v in [2, 4, 5, 10]:
    for t in enumerate_trees([2, 4, 5, 10], v):
        print("  ", t)

đệ quy thuần OK; oracle liệt kê cây khớp
các cây của [2, 4, 5, 10]:
   (2,)
   (4,)
   (4, (2,), (2,))
   (5,)
   (10,)
   (10, (2,), (5,))
   (10, (5,), (2,))


**Độ phức tạp và nút thắt.** Không nhớ nên các lời gọi `f(a)` lặp lại: với `arr` là các lũy thừa của 2, `f(2^k)` gọi `f(2^i)` cho mọi `i < k` lặp đi lặp lại, thời gian **mũ**. Hãy đếm số lần gọi:

In [9]:
def count_calls(arr):
    values = set(arr)
    calls = 0
    def f(v):
        nonlocal calls
        calls += 1
        total = 1
        for a in arr:
            if v % a == 0 and v // a in values and a < v and v // a < v:
                total += f(a) * f(v // a)
        return total
    for v in arr:
        f(v)
    return calls

for k in (3, 5, 7, 9, 11):
    arr = [2 ** i for i in range(1, k + 1)]
    print(f"arr = 2^1..2^{k} ({k} phần tử): số lần gọi f = {count_calls(arr):>12,}")

arr = 2^1..2^3 (3 phần tử): số lần gọi f =           13
arr = 2^1..2^5 (5 phần tử): số lần gọi f =          121
arr = 2^1..2^7 (7 phần tử): số lần gọi f =        1,093
arr = 2^1..2^9 (9 phần tử): số lần gọi f =        9,841
arr = 2^1..2^11 (11 phần tử): số lần gọi f =       88,573


Số lần gọi tăng gần **gấp ba** mỗi khi thêm một phần tử: bùng nổ. Nút thắt: `f(a)` chỉ phụ thuộc `a`, nên **tính một lần rồi nhớ**.

## 6. Ý tưởng thuật toán / bước đột phá

`f(v)` chỉ phụ thuộc các `f` của **ước nhỏ hơn** `v`. Do đó:

1. **Ghi nhớ** `f(v)` (Cách 2): mỗi `v` tính một lần.
2. **Sắp xếp tăng dần** rồi tính theo thứ tự đó, khi tới `v` mọi `f(a)` với `a < v` đã sẵn (Cách 3, DP bảng).
3. **Đẩy (push)** thay vì kéo (pull) (Cách 4): khi hai giá trị `a <= b` đã chốt `f` thì đẩy tích `f(a) f(b)` (nhân 2 nếu `a != b`) vào `f(a * b)` nếu `a * b` có trong `arr`.
4. **Đối xứng (Cách 5):** với `v = a * b`, mỗi cặp không thứ tự `{a, b}` cho `2 f(a) f(b)` (nếu `a != b`) hoặc `f(a)^2` (nếu `a = b`): chỉ cần duyệt `a <= isqrt(v)`.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy có nhớ (`@cache`) |
| 3 | Sắp xếp rồi kéo: `f[i] = 1 + Σ f[j] * f[idx[a[i] / a[j]]]` |
| 4 | Sắp xếp rồi đẩy theo cặp `(a, b)`, `a <= b`, tới `f(a * b)` |
| 5 | Kéo nhưng chỉ duyệt thừa số `a <= sqrt(v)` (nhân đôi cho thứ tự) |

Mọi cách có nhớ đều `O(n^2)`, với `n <= 1000` là khoảng `10^6` phép nên rất nhanh.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích từng đoạn.**

- `values = set(arr)` để kiểm tra `v // a` có trong `arr`.
- `@cache def f(v)`: `total = 1`; với mỗi `a` trong `arr` thỏa `a < v`, `v % a == 0`, `v // a` có trong `values` và `v // a < v` (luôn đúng khi `a > 1`): `total += f(a) * f(v // a)`.
- Đáp án `sum(f(v) for v in arr) % MOD`; lấy `% MOD` ngay trong `f` để số không phình.

In [10]:
def count_memo(arr: List[int]) -> int:
    values = set(arr)

    @cache
    def f(v: int) -> int:
        total = 1
        for a in arr:
            if a < v and v % a == 0 and (v // a) in values:
                total = (total + f(a) * f(v // a)) % MOD
        return total

    return sum(f(v) for v in arr) % MOD

for arr, expected in examples:
    assert count_memo(arr) == expected
print("đệ quy có nhớ OK")

đệ quy có nhớ OK


**Code nộp LeetCode (Cách 2):**

In [11]:
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        from functools import cache
        mod = 10**9 + 7
        values = set(arr)

        @cache
        def f(v: int) -> int:
            total = 1
            for a in arr:
                if a < v and v % a == 0 and (v // a) in values:
                    total = (total + f(a) * f(v // a)) % mod
            return total

        return sum(f(v) for v in arr) % mod

sol_memo = Solution().numFactoredBinaryTrees
print([sol_memo(list(a)) for a, _ in examples])

[3, 7]


**Độ phức tạp (Cách 2):** thời gian `O(n^2)` (mỗi `v` duyệt `n` giá trị `a`, tra `dict` `O(1)`); bộ nhớ `O(n)` cho bộ nhớ đệm cộng độ sâu đệ quy tối đa bằng độ dài chuỗi ước (`<= 30` vì mỗi bước ít nhất chia đôi).

---

### 7.2. Cách 3 — DP sắp xếp (kéo, lời giải chuẩn)

**Giải thích từng đoạn.**

- `arr.sort()`; `idx = {giá trị: chỉ số}`; `f = [1] * n` (mỗi giá trị là một cây một nút).
- Với `i` từ `0` tới `n - 1`, `j` từ `0` tới `i - 1`: đặt `b = arr[j]` (một thừa số nhỏ hơn). Nếu `arr[i] % b == 0` và `c = arr[i] // b` có trong `idx`: `f[i] += f[j] * f[idx[c]]`. Vì ta duyệt **mọi** `j` (cả `b` làm con trái lẫn con phải khi `c != b`), mỗi cặp có thứ tự được tính đúng một lần.
- Đáp án `sum(f) % MOD`.

In [12]:
def count_dp(arr: List[int]) -> int:
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (c := a // b) in idx:
                f[i] = (f[i] + f[j] * f[idx[c]]) % MOD
    return sum(f) % MOD

for arr, expected in examples:
    assert count_dp(arr) == expected
print("DP kéo OK")

DP kéo OK


**Code nộp LeetCode (Cách 3):**

In [13]:
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        n = len(arr)
        arr.sort()
        idx = {v: i for i, v in enumerate(arr)}
        f = [1] * n
        for i, a in enumerate(arr):
            for j in range(i):
                b = arr[j]
                if a % b == 0 and (c := (a // b)) in idx:
                    f[i] = (f[i] + f[j] * f[idx[c]]) % mod
        return sum(f) % mod

sol_dp = Solution().numFactoredBinaryTrees
print([sol_dp(list(a)) for a, _ in examples])

[3, 7]


**Độ phức tạp (Cách 3):** thời gian `O(n log n + n^2)`, bộ nhớ `O(n)`. **Đây là cách nên nộp.**

---

### 7.3. Cách 4 — DP đẩy theo cặp `(a, b)`

**Ý tưởng khác.** Đảo chiều: sau khi `f(a)` và `f(b)` đã **chốt** (mọi đóng góp vào chúng đã được cộng), đẩy `f(a) * f(b)` (nhân 2 nếu `a != b`) vào `f(a * b)` nếu `a * b` có trong `arr`.

**Giải thích từng đoạn.**

- Sắp xếp tăng dần, `f = [1] * n`.
- Duyệt `i` tăng dần (`b = arr[i]`, thừa số **lớn** của cặp) và `j` từ `0` tới `i` (`a = arr[j] <= b`): `p = a * b`; nếu `p` có trong `idx`: `f[idx[p]] += f[j] * f[i] * (1 nếu i == j, ngược lại 2)`.
- Vì sao đúng thứ tự: khi ta xử lý `i`, mọi đóng góp vào `f[i]` đến từ các cặp có tích `arr[i]` với cả hai thừa số **nhỏ hơn** `arr[i]`, tức là cặp có thừa số lớn có chỉ số `< i`, đã được đẩy ở các bước trước. Vậy `f[i]` đã chốt khi ta dùng nó.

In [14]:
def count_push(arr: List[int]) -> int:
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i in range(n):
        b = arr[i]
        for j in range(i + 1):
            a = arr[j]
            k = idx.get(a * b)
            if k is not None:
                f[k] = (f[k] + f[j] * f[i] * (1 if i == j else 2)) % MOD
    return sum(f) % MOD

for arr, expected in examples:
    assert count_push(arr) == expected
print("DP đẩy OK")

DP đẩy OK


**Code nộp LeetCode (Cách 4):**

In [15]:
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        arr = sorted(arr)
        n = len(arr)
        idx = {v: i for i, v in enumerate(arr)}
        f = [1] * n
        for i in range(n):
            b = arr[i]
            for j in range(i + 1):
                k = idx.get(arr[j] * b)
                if k is not None:
                    f[k] = (f[k] + f[j] * f[i] * (1 if i == j else 2)) % mod
        return sum(f) % mod

sol_push = Solution().numFactoredBinaryTrees
print([sol_push(list(a)) for a, _ in examples])

[3, 7]


**Độ phức tạp (Cách 4):** thời gian `O(n^2)` (khoảng một nửa so với Cách 3 vì chỉ duyệt `j <= i`), bộ nhớ `O(n)`. Chú ý: phép nhân `a * b` có thể tới `10^18` (Python không tràn; trong Java/C++ phải dùng `long`).

---

### 7.4. Cách 5 — Kéo nhưng chỉ duyệt thừa số `<= sqrt(v)`

**Ý tưởng khác.** Cặp `(a, b)` với `a * b = v` và `a < b` xuất hiện hai lần (có thứ tự); ta duyệt **chỉ thừa số nhỏ** `a <= isqrt(v)`: nếu `a == b` cộng `f(a)^2`; nếu `a != b` cộng `2 f(a) f(b)`.

**Giải thích từng đoạn.**

- Sắp xếp; với mỗi `v = arr[i]`: lặp `j` tăng dần với `a = arr[j]`; dừng khi `a * a > v`.
- Nếu `v % a == 0` và `b = v // a` có trong `idx`: cộng `f[j] * f[idx[b]]` (nhân 2 nếu `a != b`).
- Dừng sớm giúp bỏ qua khoảng nửa các `j` (và nhiều hơn khi `v` nhỏ).

In [16]:
def count_sqrt(arr: List[int]) -> int:
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, v in enumerate(arr):
        total = 1
        for j in range(i):
            a = arr[j]
            if a * a > v:
                break
            if v % a == 0:
                k = idx.get(v // a)
                if k is not None:
                    total += f[j] * f[k] * (1 if a * a == v else 2)
        f[i] = total % MOD
    return sum(f) % MOD

for arr, expected in examples:
    assert count_sqrt(arr) == expected
print("DP đối xứng OK")

DP đối xứng OK


**Code nộp LeetCode (Cách 5):**

In [17]:
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        arr = sorted(arr)
        n = len(arr)
        idx = {v: i for i, v in enumerate(arr)}
        f = [1] * n
        for i, v in enumerate(arr):
            total = 1
            for j in range(i):
                a = arr[j]
                if a * a > v:
                    break
                if v % a == 0:
                    k = idx.get(v // a)
                    if k is not None:
                        total += f[j] * f[k] * (1 if a * a == v else 2)
            f[i] = total % mod
        return sum(f) % mod

sol_sqrt = Solution().numFactoredBinaryTrees
print([sol_sqrt(list(a)) for a, _ in examples])

[3, 7]


**Độ phức tạp (Cách 5):** thời gian `O(n^2)` tệ nhất nhưng hằng số nhỏ (dừng sớm khi `a * a > v`), bộ nhớ `O(n)`.

## 8. TRACE từng bước

### 8.1. Cách 3 trên `arr = [2, 4, 5, 10]`

Sau sắp xếp `[2, 4, 5, 10]`, `idx = {2: 0, 4: 1, 5: 2, 10: 3}`, `f = [1, 1, 1, 1]`.

| `i` | `a = arr[i]` | `j` | `b = arr[j]` | `a % b == 0`? | `c = a // b` có trong `idx`? | cập nhật `f[i]` | `f` sau |
|---|---|---|---|---|---|---|---|
| 1 | 4 | 0 | 2 | có | `c = 2` có | `1 + f[0] * f[0] = 2` | `[1, 2, 1, 1]` |
| 2 | 5 | 0, 1 | 2, 4 | không | | | `[1, 2, 1, 1]` |
| 3 | 10 | 0 | 2 | có | `c = 5` có (chỉ số 2) | `1 + f[0] * f[2] = 2` | `[1, 2, 1, 2]` |
| 3 | 10 | 1 | 4 | không | | | |
| 3 | 10 | 2 | 5 | có | `c = 2` có (chỉ số 0) | `2 + f[2] * f[0] = 3` | `[1, 2, 1, 3]` |

Tổng `1 + 2 + 1 + 3 = 7`. Cặp `(2, 5)` và `(5, 2)` được cộng ở hai giá trị `j` khác nhau.

In [18]:
def trace_dp(arr):
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    print("arr đã sắp xếp:", arr, "| idx:", idx, "| f ban đầu:", f)
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx:
                c = a // b
                old = f[i]
                f[i] += f[j] * f[idx[c]]
                print(f"  a={a}, b={b}, c={c}: f[{i}] = {old} + f[{j}] * f[{idx[c]}] = {old} + {f[j]} * {f[idx[c]]} = {f[i]}")
        print(f"xong i={i} (a={a}): f = {f}")
    print("tổng =", sum(f))
    return sum(f)

assert trace_dp([2, 4, 5, 10]) == 7

arr đã sắp xếp: [2, 4, 5, 10] | idx: {2: 0, 4: 1, 5: 2, 10: 3} | f ban đầu: [1, 1, 1, 1]
xong i=0 (a=2): f = [1, 1, 1, 1]
  a=4, b=2, c=2: f[1] = 1 + f[0] * f[0] = 1 + 1 * 1 = 2
xong i=1 (a=4): f = [1, 2, 1, 1]
xong i=2 (a=5): f = [1, 2, 1, 1]
  a=10, b=2, c=5: f[3] = 1 + f[0] * f[2] = 1 + 1 * 1 = 2
  a=10, b=5, c=2: f[3] = 2 + f[2] * f[0] = 2 + 1 * 1 = 3
xong i=3 (a=10): f = [1, 2, 1, 3]
tổng = 7


### 8.2. Cách 4 (đẩy) trên `[2, 4, 5, 10]`

Các cặp `(a, b)` với `a <= b` và `a * b` trong `arr`: `(2, 2) -> 4` (nhân 1), `(2, 5) -> 10` (nhân 2). `f[4] = 1 + 1 * 1 * 1 = 2`; `f[10] = 1 + f[2] * f[5] * 2 = 3`.

In [19]:
def trace_push(arr):
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i in range(n):
        b = arr[i]
        for j in range(i + 1):
            a = arr[j]
            k = idx.get(a * b)
            if k is not None:
                mult = 1 if i == j else 2
                f[k] += f[j] * f[i] * mult
                print(f"  cặp ({a}, {b}) có tích {a * b} trong arr: f[{k}] += f[{j}] * f[{i}] * {mult} -> f[{k}] = {f[k]}")
    print("f =", f, "tổng =", sum(f))
    return sum(f)

assert trace_push([2, 4, 5, 10]) == 7

  cặp (2, 2) có tích 4 trong arr: f[1] += f[0] * f[0] * 1 -> f[1] = 2
  cặp (2, 5) có tích 10 trong arr: f[3] += f[0] * f[2] * 2 -> f[3] = 3
f = [1, 2, 1, 3] tổng = 7


### 8.3. Cách 5: dừng sớm khi `a * a > v`

Với `v = 10`: `a = 2` (`4 <= 10`, `10 % 2 == 0`, `b = 5` có -> `f[2] * f[5] * 2` vì `2 != 5`); `a = 4` (`16 > 10`) dừng.

In [20]:
def trace_sqrt(arr):
    arr = sorted(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * len(arr)
    for i, v in enumerate(arr):
        total = 1
        for j in range(i):
            a = arr[j]
            if a * a > v:
                print(f"  v={v}: a={a}, a*a={a * a} > v -> dừng")
                break
            if v % a == 0 and (v // a) in idx:
                k = idx[v // a]
                mult = 1 if a * a == v else 2
                total += f[j] * f[k] * mult
                print(f"  v={v}: a={a}, b={v // a}, nhân {mult}: total = {total}")
        f[i] = total
    print("f =", f, "tổng =", sum(f))
    return sum(f)

assert trace_sqrt([2, 4, 5, 10]) == 7

  v=4: a=2, b=2, nhân 1: total = 2
  v=5: a=4, a*a=16 > v -> dừng
  v=10: a=2, b=5, nhân 2: total = 3
  v=10: a=4, a*a=16 > v -> dừng
f = [1, 2, 1, 3] tổng = 7


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (DP sắp xếp, kéo): ngắn, dễ kiểm, `O(n^2)` đủ cho `n <= 1000`.

```python
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        n = len(arr)
        arr.sort()
        idx = {v: i for i, v in enumerate(arr)}
        f = [1] * n
        for i, a in enumerate(arr):
            for j in range(i):
                b = arr[j]
                if a % b == 0 and (c := (a // b)) in idx:
                    f[i] = (f[i] + f[j] * f[idx[c]]) % mod
        return sum(f) % mod
```

In [21]:
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        n = len(arr)
        arr.sort()
        idx = {v: i for i, v in enumerate(arr)}
        f = [1] * n
        for i, a in enumerate(arr):
            for j in range(i):
                b = arr[j]
                if a % b == 0 and (c := (a // b)) in idx:
                    f[i] = (f[i] + f[j] * f[idx[c]]) % mod
        return sum(f) % mod

sol = Solution()
for arr, expected in examples:
    got = sol.numFactoredBinaryTrees(list(arr))
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[2, 4] -> 3 OK
[2, 4, 5, 10] -> 7 OK


## 10. Vì sao đúng

**(a) Công thức.** Gọi `T(v)` là tập các cây có gốc `v`. Một cây thuộc `T(v)` hoặc là nút đơn `v` (1 cây), hoặc là nút `v` với con trái có giá trị `a`, con phải có giá trị `b`, `a * b = v`, `a, b ∈ arr`, cùng hai cây con tùy ý thuộc `T(a)` và `T(b)`. Hai cây khác nhau nếu khác nút đơn/nút trong, khác cặp `(a, b)` có thứ tự, hoặc khác một cây con. Vậy các trường hợp **rời nhau** và `|T(v)| = 1 + Σ_{(a, b) có thứ tự, a b = v} |T(a)| * |T(b)|` (nguyên lý cộng và nhân). Đáp án là `Σ_v |T(v)|`.

**(b) Thứ tự tính.** Mọi phần tử `> 1` nên `a, b < v` khi `a * b = v` (vì `b >= 2` suy ra `a = v / b <= v / 2 < v`). Do đó `f(v)` chỉ dùng `f` của các giá trị **nhỏ hơn** `v`: sắp xếp tăng dần và tính theo thứ tự đó là hợp lệ (Cách 3). Cách 2 có nhớ cho cùng kết quả vì `f(v)` chỉ phụ thuộc `v`.

**(c) Cách 3 duyệt đủ cặp có thứ tự.** Với mỗi `v = arr[i]` và mỗi `j < i`, `b = arr[j]` đóng vai **một thừa số**; `c = v / b`. Cặp có thứ tự `(a, b) = (c, b)` nếu ta coi `b` là con phải, hoặc `(b, c)` nếu `b` là con trái. Khi `c != b` cả hai cặp có thứ tự `(b, c)` và `(c, b)` được cộng, ở hai giá trị `j` khác nhau (`b` và `c` đều xuất hiện trong `arr[0..i-1]`). Khi `c == b` chỉ một lần. Vậy mỗi cặp có thứ tự được cộng đúng một lần.

**(d) Cách 4.** Cặp không thứ tự `{a, b}` với `a < b` đóng góp `2 f(a) f(b)` (hai thứ tự); `a = b` đóng góp `f(a)^2`. Thời điểm đẩy (`i` là chỉ số của `b >= a`): `f[i]` đã chốt vì mọi đóng góp vào `arr[i]` có hai thừa số nhỏ hơn `arr[i]`, tức thừa số lớn hơn chỉ số `< i`.

**(e) Cách 5.** Cùng công thức nhưng gộp hai thứ tự thành hệ số 2 và chỉ duyệt thừa số nhỏ `a <= sqrt(v)`.

**(f) Modulo.** Phép cộng và nhân đều tương thích với `% MOD` (mục 3.4), nên lấy `% MOD` giữa chừng cho kết quả đồng dư với kết quả chính xác. Ta kiểm tra bằng số nguyên lớn (không modulo).

**Kiểm chứng bằng máy:** (1) so mọi cách với oracle **liệt kê cây** trên **mọi** tập con của `{2..16}` (và các tập có giá trị lớn hơn nhưng nhiều quan hệ nhân); (2) so kết quả có modulo với kết quả **chính xác không modulo** lấy `%` ở cuối trên các mảng lớn.

In [22]:
def count_exact(arr: List[int]) -> int:
    """Cùng công thức nhưng KHÔNG modulo (số nguyên lớn chính xác)."""
    arr = sorted(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * len(arr)
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx:
                f[i] += f[j] * f[idx[a // b]]
    return sum(f)

solutions = {
    "rec": count_rec,
    "memo": count_memo,
    "dp": count_dp,
    "push": count_push,
    "sqrt": count_sqrt,
    "sol_memo": lambda a: sol_memo(list(a)),
    "sol_dp": lambda a: sol_dp(list(a)),
    "sol_push": lambda a: sol_push(list(a)),
    "sol_sqrt": lambda a: sol_sqrt(list(a)),
    "final": lambda a: Solution().numFactoredBinaryTrees(list(a)),
}

# mọi tập con (không rỗng) của {2, ..., 17}: 2^16 - 1 = 65535 tập nhưng oracle liệt kê cây chậm -> dùng {2..12} (2047 tập)
checked = 0
universe = list(range(2, 13))
for mask in range(1, 1 << len(universe)):
    arr = [universe[t] for t in range(len(universe)) if mask >> t & 1]
    ref = oracle(arr)
    assert ref == count_exact(arr) % MOD, arr
    for name, fn in solutions.items():
        assert fn(list(arr)) == ref, (name, arr, fn(list(arr)), ref)
    checked += 1
print(f"mọi tập con không rỗng của {{2..12}}: {checked} tập x {len(solutions)} cài đặt khớp oracle liệt kê cây")

mọi tập con không rỗng của {2..12}: 2047 tập x 10 cài đặt khớp oracle liệt kê cây


## 11. Time / Space Complexity

Gọi `n = len(arr)` (`<= 1000`).

| Cách | Time | Space |
|---|---|---|
| 1. Đệ quy thuần | mũ | `O(độ sâu)` |
| 2. Đệ quy có nhớ | `O(n^2)` | `O(n)` |
| 3. DP sắp xếp (kéo) | `O(n log n + n^2)` | `O(n)` |
| 4. DP đẩy | `O(n log n + n^2 / 2)` | `O(n)` |
| 5. DP đối xứng `a <= sqrt(v)` | `O(n^2)` tệ nhất, thường ít hơn | `O(n)` |

Với `n = 1000`: khoảng `10^6` phép; vài chục mili-giây. Đo trên các tập khó (nhiều quan hệ nhân):

In [23]:
def smooth_numbers(limit, primes, count):
    vals = {1}
    for p in primes:
        vals |= {v * p ** e for v in list(vals) for e in range(1, 31) if v * p ** e <= limit}
    vals.discard(1)
    return sorted(vals)[:count]

rng = random.Random(823)
tests = []
smooth = smooth_numbers(10 ** 9, [2, 3, 5, 7], 4000)
tests.append(rng.sample(smooth, 1000))                                 # nhiều quan hệ nhân
tests.append(smooth[:1000])                                            # các số trơn nhỏ nhất: rất nhiều cặp tích có trong tập
tests.append([2 ** i for i in range(1, 30)])                           # lũy thừa của 2
tests.append(rng.sample(range(2, 10 ** 9), 1000))                      # ngẫu nhiên: gần như không có quan hệ
tests.append(list(range(2, 1002)))                                     # 2..1001: mọi quan hệ nhân thực sự

ref = None
for name in ["memo", "dp", "push", "sqrt"]:
    t0 = time.perf_counter()
    res = [solutions[name](list(a)) for a in tests]
    dt = (time.perf_counter() - t0) * 1000
    if ref is None:
        ref = res
    assert res == ref, name
    print(f"{name:5} {len(tests)} test (n tới 1000): {dt:8.1f} ms")
print("kết quả:", ref)
for a, r in zip(tests, ref):
    assert count_exact(a) % MOD == r
print("kiểm tra với số nguyên lớn không modulo: khớp")

memo  5 test (n tới 1000):    252.9 ms
dp    5 test (n tới 1000):    138.7 ms


push  5 test (n tới 1000):    161.9 ms
sqrt  5 test (n tới 1000):     19.7 ms
kết quả: [159233, 2812872, 93720241, 1000, 769795]
kiểm tra với số nguyên lớn không modulo: khớp


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với oracle liệt kê cây (mảng nhỏ) và với **số nguyên lớn không modulo** (mảng lớn), và test ngẫu nhiên có seed (đúng ràng buộc: giá trị phân biệt, `2 <= arr[i] <= 10^9`, `n <= 1000`).

In [24]:
# 12.1 ví dụ đề
for arr, expected in examples:
    for name, fn in solutions.items():
        assert fn(list(arr)) == expected, (name, arr)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay; ba ca cuối lấy từ oracle liệt kê cây)
edge = [
    ([2], 1),
    ([10 ** 9], 1),
    ([2, 3, 5], 3),
    ([2, 3, 6], 5),
    ([2, 4, 8], 8),
    ([4, 2], 3),
    ([2, 4, 5, 10], 7),
    ([3, 9, 27], 8),
    ([2, 4, 8, 16], oracle([2, 4, 8, 16])),
    ([6, 2, 3, 12], oracle([6, 2, 3, 12])),
    ([2, 4, 5, 10, 20, 100], oracle([2, 4, 5, 10, 20, 100])),
]
for arr, expected in edge:
    assert count_exact(arr) % MOD == expected, ("số nguyên lớn khác kỳ vọng", arr)
    for name, fn in solutions.items():
        assert fn(list(arr)) == expected, (name, arr, fn(list(arr)), expected)
print("12.2 edge case: OK", len(edge), "ca; giá trị oracle:", [e for _, e in edge[-3:]])

12.1 ví dụ đề: OK
12.2 edge case: OK 11 ca; giá trị oracle: [23, 12, 50]


In [25]:
# 12.3 phản ví dụ: các cách sai
def wrong_no_sort(arr):                 # quên sắp xếp
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx:
                f[i] += f[j] * f[idx[a // b]]
    return sum(f) % MOD

def wrong_no_single(arr):               # quên cây một nút
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [0] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx:
                f[i] += f[j] * f[idx[a // b]]
    return sum(f) % MOD

def wrong_unordered(arr):               # chỉ đếm một thứ tự con (không nhân đôi khi a != b)
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx and b <= a // b:
                f[i] += f[j] * f[idx[a // b]]
    return sum(f) % MOD

def wrong_no_check_divisible(arr):      # dùng // mà không kiểm tra v % b == 0
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            c = a // b                  # 5 // 2 = 2 (làm tròn xuống) được coi như hợp lệ
            if c in idx:
                f[i] += f[j] * f[idx[c]]
    return sum(f) % MOD

wrongs = [
    ("quên sắp xếp", wrong_no_sort, [10, 5, 2]),
    ("quên cây một nút", wrong_no_single, [2, 4]),
    ("chỉ một thứ tự con", wrong_unordered, [2, 4, 5, 10]),
    ("không kiểm tra chia hết", wrong_no_check_divisible, [2, 5]),
]
for name, fn, arr in wrongs:
    truth = oracle(arr)
    print(f"{name:26} arr={arr}: sai = {fn(arr)}, đúng = {truth}")
    assert fn(arr) != truth, name
print("12.3 phản ví dụ: OK (các cách sai đều bị lộ)")

quên sắp xếp               arr=[10, 5, 2]: sai = 3, đúng = 5
quên cây một nút           arr=[2, 4]: sai = 0, đúng = 3
chỉ một thứ tự con         arr=[2, 4, 5, 10]: sai = 6, đúng = 7
không kiểm tra chia hết    arr=[2, 5]: sai = 3, đúng = 2
12.3 phản ví dụ: OK (các cách sai đều bị lộ)


In [26]:
# 12.4 ngẫu nhiên nhỏ so với oracle liệt kê cây
rng = random.Random(2823)
for _ in range(300):
    n = rng.randint(1, 6)
    pool = rng.choice([list(range(2, 25)), [2, 3, 4, 6, 8, 9, 12, 16, 18, 24, 27, 36, 48, 54, 72, 81, 108]])
    arr = rng.sample(pool, min(n, len(pool)))
    ref = oracle(arr)
    for name, fn in solutions.items():
        assert fn(list(arr)) == ref, (name, arr)
print("12.4 ngẫu nhiên nhỏ so với oracle: OK")

# 12.5 ngẫu nhiên lớn: các cách khớp nhau và khớp số nguyên lớn không modulo (đúng ràng buộc)
rng = random.Random(77)
big_cases = 0
for _ in range(30):
    n = rng.randint(50, 1000)
    mode = rng.choice(["smooth", "range", "random", "powers"])
    if mode == "smooth":
        arr = rng.sample(smooth, min(n, len(smooth)))
    elif mode == "range":
        start = rng.randint(2, 500)
        arr = list(range(start, start + n))
    elif mode == "random":
        arr = rng.sample(range(2, 10 ** 9 + 1), n)
    else:
        arr = [2 ** i for i in range(1, 30)] + [3 ** i for i in range(1, 19)]
        arr = list(dict.fromkeys(arr))
    res = {name: fn(list(arr)) for name, fn in solutions.items() if name != "rec"}
    assert len(set(res.values())) == 1, (mode, n, res)
    assert count_exact(arr) % MOD == res["final"], (mode, n)
    big_cases += 1
print(f"12.5 ngẫu nhiên lớn: OK ({big_cases} ca, so với số nguyên lớn không modulo)")

# 12.6 tính chất: kết quả chỉ phụ thuộc tập hợp (không phụ thuộc thứ tự trong mảng)
rng = random.Random(6)
for _ in range(100):
    arr = rng.sample(smooth, rng.randint(1, 60))
    shuffled = arr[:]
    rng.shuffle(shuffled)
    assert Solution().numFactoredBinaryTrees(list(arr)) == Solution().numFactoredBinaryTrees(shuffled)
print("12.6 kết quả không phụ thuộc thứ tự mảng: OK")

12.4 ngẫu nhiên nhỏ so với oracle: OK


12.5 ngẫu nhiên lớn: OK (30 ca, so với số nguyên lớn không modulo)
12.6 kết quả không phụ thuộc thứ tự mảng: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp mảng

Công thức `f(v)` cần `f` của các ước nhỏ hơn **đã tính xong**. Nếu mảng chưa sắp xếp, khi tới `v` thì `f(a)` có thể còn là giá trị khởi tạo.

In [27]:
arr = [10, 5, 2]
print("sai (không sắp xếp):", wrong_no_sort(arr), "| đúng:", Solution().numFactoredBinaryTrees(list(arr)))
assert wrong_no_sort(arr) == 3 and Solution().numFactoredBinaryTrees(list(arr)) == 5

sai (không sắp xếp): 3 | đúng: 5


Vì sao sai: với `[10, 5, 2]` ta xử lý `10` đầu tiên khi `j` chỉ chạy trên các chỉ số nhỏ hơn: không có ước nào. **Đúng:** `arr.sort()` trước.

### Lỗi 2 — Quên cây một nút

Mỗi giá trị tự nó là một cây (lá). Khởi tạo `f = [0] * n` thì mọi tích đều là 0.

In [28]:
arr = [2, 4]
print("sai (f khởi tạo 0):", wrong_no_single(arr), "| đúng:", Solution().numFactoredBinaryTrees(list(arr)))
assert wrong_no_single(arr) == 0 and Solution().numFactoredBinaryTrees(list(arr)) == 3

sai (f khởi tạo 0): 0 | đúng: 3


Vì sao sai: không có cây đơn thì cũng không dựng được cây lớn. **Đúng:** `f = [1] * n`.

### Lỗi 3 — Chỉ đếm một thứ tự con

`[10, 2, 5]` và `[10, 5, 2]` là hai cây khác nhau. Chỉ đếm cặp `b <= c` bỏ sót nửa.

In [29]:
arr = [2, 4, 5, 10]
print("sai (một thứ tự):", wrong_unordered(arr), "| đúng:", Solution().numFactoredBinaryTrees(list(arr)))
assert wrong_unordered(arr) == 6 and Solution().numFactoredBinaryTrees(list(arr)) == 7

sai (một thứ tự): 6 | đúng: 7


Vì sao sai: đề phân biệt con trái/phải. **Đúng:** cộng cả `(b, c)` và `(c, b)` khi `b != c` (Cách 3 làm tự nhiên vì duyệt mọi `j`).

### Lỗi 4 — Không kiểm tra chia hết

`a // b` làm tròn xuống; nếu `a % b != 0` thì `a // b` **không phải** thừa số thật sự.

In [30]:
arr = [2, 5]
print("sai (không kiểm tra chia hết):", wrong_no_check_divisible(arr), "| đúng:", Solution().numFactoredBinaryTrees(list(arr)))
assert wrong_no_check_divisible(arr) != Solution().numFactoredBinaryTrees(list(arr))

sai (không kiểm tra chia hết): 3 | đúng: 2


Vì sao sai: `5 // 2 = 2` trông như "5 = 2 * 2" nhưng thực tế `2 * 2 = 4 != 5`. **Đúng:** `a % b == 0` trước rồi mới lấy `a // b`.

### Lỗi 5 — Quên modulo (hoặc không nhận ra đáp án rất lớn)

Với `arr` là các lũy thừa của 2 đến `2^29`, đáp án chính xác vượt xa `10^9 + 7`. Python không tràn nhưng đề yêu cầu **kết quả theo modulo**.

In [31]:
arr = [2 ** i for i in range(1, 30)]
exact = count_exact(arr)
print("đáp án chính xác (không modulo):", exact)
print("đáp án theo modulo           :", exact % MOD, "| nộp:", Solution().numFactoredBinaryTrees(list(arr)))
assert exact > MOD and Solution().numFactoredBinaryTrees(list(arr)) == exact % MOD

đáp án chính xác (không modulo): 243223512796284818
đáp án theo modulo           : 93720241 | nộp: 93720241


Vì sao quan trọng: trả `exact` sẽ sai ở LeetCode; Java/C++ thì tràn số nếu không lấy modulo sau mỗi phép cộng/nhân (và tích hai số `< 10^9 + 7` cần kiểu `long`). **Đúng:** `% mod` sau mỗi cập nhật và ở tổng cuối.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "đếm cấu trúc dựng từ các thành phần nhỏ hơn", công thức "1 + tổng tích" -> DP theo thứ tự tăng dần (số nhỏ trước).
- **Nguyên lý nhân/cộng:** cây con trái/phải độc lập -> nhân; các dạng rời nhau -> cộng.
- **Thứ tự quan trọng:** `(a, b)` và `(b, a)` khác nhau khi `a != b`.
- **Dict giá trị -> chỉ số** giúp kiểm tra "`v / a` có trong tập không" trong `O(1)`.
- **Modulo:** lấy `% mod` mỗi lần cộng/nhân; kiểm lại bằng số nguyên lớn khi viết bằng Python.
- Giới hạn `n <= 1000` cho phép `O(n^2)`; giá trị tới `10^9` nên không thể dùng bảng theo giá trị.
- Cùng họ: Count Number of Special Subsequences, Number of Ways to Form a Target String, các bài đếm cây theo tích/tổng (Unique Binary Search Trees 96).
- Test nên có: một phần tử, không có quan hệ nhân, `[2, 3, 6]` (hai thứ tự), lũy thừa của 2.

## 15. Biến thể

1. **Đếm số cây theo từng gốc**: trả về dict `giá trị -> f(giá trị)`.
2. **Liệt kê các cây** (đã làm trong oracle `enumerate_trees`).
3. **Giới hạn chiều cao** (bài tập 5): `f_h(v)` là số cây gốc `v` có chiều cao tối đa `h`.
4. **Cây có nhiều hơn hai con** (mỗi nút trong bằng tích các con, số con >= 2): khó hơn, cần phân hoạch nhân.
5. Bài liên quan: Unique Binary Search Trees (96), Different Ways to Add Parentheses (241).

Dưới đây là code cho biến thể 1, kiểm bằng oracle liệt kê.

In [32]:
def count_by_root(arr: List[int]) -> dict:
    arr = sorted(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * len(arr)
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (a // b) in idx:
                f[i] = (f[i] + f[j] * f[idx[a // b]]) % MOD
    return dict(zip(arr, f))

print(count_by_root([2, 4, 5, 10]))
assert count_by_root([2, 4, 5, 10]) == {2: 1, 4: 2, 5: 1, 10: 3}
rng = random.Random(15)
for _ in range(100):
    arr = rng.sample(list(range(2, 40)), rng.randint(1, 6))
    for v, c in count_by_root(arr).items():
        assert c == len(enumerate_trees(arr, v)) % MOD, (arr, v)
print("Biến thể: OK")

{2: 1, 4: 2, 5: 1, 10: 3}
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | Công thức, không nhớ | mũ | `O(độ sâu)` | Chỉ để hiểu công thức |
| 2. Đệ quy có nhớ | `@cache` theo giá trị | `O(n^2)` | `O(n)` | Dễ suy ra từ công thức |
| 3. DP sắp xếp (kéo) | `f[i] = 1 + Σ f[j] f[idx[a/b]]` | `O(n^2)` | `O(n)` | **Cách nộp chính** |
| 4. DP đẩy theo cặp | Đẩy `f(a) f(b)` (nhân 2) sang `f(a b)` | `O(n^2/2)` | `O(n)` | Khi muốn bớt chia/modulo; dễ hiểu theo cặp |
| 5. Đối xứng `<= sqrt` | Chỉ thừa số nhỏ, nhân đôi | `O(n^2)` nhỏ hơn | `O(n)` | Tối ưu hằng số |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(sorted([10, 2, 5]))
print({v: i for i, v in enumerate([2, 4, 5])})
print(Solution().numFactoredBinaryTrees([2, 4]))
```

In [33]:
# Bài 1: ghi dự đoán của bạn (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [2, 4, 8]`: điền `f(2)`, `f(4)`, `f(8)` (số cây có gốc tương ứng) và tổng.

In [34]:
# Bài 2: điền kết quả tính tay
hand_f = None        # dict {2: ..., 4: ..., 8: ...}
hand_total = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành DP sắp xếp.

In [35]:
def exercise_3_dp(arr: List[int]) -> int:
    mod = 10 ** 9 + 7
    # TODO 1: sắp xếp arr; tạo idx: giá trị -> chỉ số
    # TODO 2: f = [1] * n (mỗi giá trị là một cây một nút)
    # TODO 3: với mỗi i, với mỗi j < i: nếu arr[i] % arr[j] == 0 và arr[i] // arr[j] có trong idx
    #         thì f[i] += f[j] * f[idx[arr[i] // arr[j]]]  (nhớ % mod)
    # TODO 4: trả về tổng f theo modulo
    return None


# assert exercise_3_dp([2, 4]) == 3
# assert exercise_3_dp([2, 4, 5, 10]) == 7
# assert exercise_3_dp([2, 4, 8]) == 8

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_push(arr)`: duyệt các cặp `a <= b` đã chốt và đẩy `f(a) * f(b)` (nhân 2 nếu `a != b`) vào `f(a * b)`.

In [36]:
def exercise_4_push(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_push([2, 4]) == 3
# assert exercise_4_push([2, 4, 5, 10]) == 7
# assert exercise_4_push([2, 3, 6]) == 5

### Bài 5 — Mini challenge: cây có chiều cao tối đa `h` (khó)

Đếm số cây nhị phân hợp lệ (như đề) có **chiều cao không quá `h`** (cây một nút có chiều cao 1). Kết quả theo modulo `10^9 + 7`.

```python
exercise_5_height([2, 4, 5, 10], 1)  -> 4      (chỉ các nút đơn)
exercise_5_height([2, 4, 5, 10], 2)  -> 7
exercise_5_height([2, 4, 8], 2)      -> 1 + 2 + 3 = 6   # f_2(8) = 1 + 2 * f_1(2) * f_1(4) = 3
exercise_5_height([2, 4, 8], 3)      -> 8
```
Gợi ý: `f_1(v) = 1`; `f_h(v) = 1 + Σ f_{h-1}(a) * f_{h-1}(b)` trên các cặp có thứ tự `a * b = v`.

In [37]:
def exercise_5_height(arr: List[int], h: int) -> int:
    # TODO
    pass


# assert exercise_5_height([2, 4, 5, 10], 1) == 4
# assert exercise_5_height([2, 4, 5, 10], 2) == 7
# assert exercise_5_height([2, 4, 8], 2) == 6
# assert exercise_5_height([2, 4, 8], 3) == 8

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `sorted` trả list tăng dần; dict comprehension ánh xạ giá trị -> chỉ số.
- Mức 2: `[2, 4]` có `f(2) = 1`, `f(4) = 2`.
- Mức 3: `[2, 5, 10]`, `{2: 0, 4: 1, 5: 2}`, `3`.

**Bài 2**
- Mức 1: `f(2) = 1` (không có thừa số nhỏ hơn).
- Mức 2: `f(4) = 1 + f(2) * f(2) = 2`; `8 = 2 * 4 = 4 * 2` (hai thứ tự).
- Mức 3: `f(8) = 1 + 2 * f(2) * f(4) = 5`; tổng `1 + 2 + 5 = 8`.

**Bài 3**
- Mức 1: `arr.sort()`, `idx = {v: i for i, v in enumerate(arr)}`.
- Mức 2: vòng `for i, a in enumerate(arr): for j in range(i): b = arr[j]`.
- Mức 3: `if a % b == 0 and (c := a // b) in idx: f[i] = (f[i] + f[j] * f[idx[c]]) % mod`; `return sum(f) % mod`.

**Bài 4**
- Mức 1: `for i in range(n): for j in range(i + 1):` với `a = arr[j]`, `b = arr[i]`.
- Mức 2: `k = idx.get(a * b)`; nếu có thì cộng.
- Mức 3: `f[k] += f[j] * f[i] * (1 if i == j else 2)`.

**Bài 5**
- Mức 1: tính theo từng mức `h`: `prev` là mảng `f_{h-1}`; `f_1 = [1] * n`.
- Mức 2: `cur[i] = 1 + Σ prev[j] * prev[idx[c]]` trên `j < i` với `arr[i] = arr[j] * c`.
- Mức 3: lặp `h - 1` lần; đáp án `sum(cur) % mod`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [38]:
print(sorted([10, 2, 5]))
print({v: i for i, v in enumerate([2, 4, 5])})
print(Solution().numFactoredBinaryTrees([2, 4]))

assert sorted([10, 2, 5]) == [2, 5, 10]
assert {v: i for i, v in enumerate([2, 4, 5])} == {2: 0, 4: 1, 5: 2}
assert Solution().numFactoredBinaryTrees([2, 4]) == 3

[2, 5, 10]
{2: 0, 4: 1, 5: 2}
3


### Đáp án Bài 2

`f(2) = 1`, `f(4) = 2`, `f(8) = 1 + 2 * f(2) * f(4) = 5` (hai thứ tự `(2, 4)` và `(4, 2)`); tổng `8`.

In [39]:
hand_f_solution = {2: 1, 4: 2, 8: 5}
hand_total_solution = 8
assert count_by_root([2, 4, 8]) == hand_f_solution
assert sum(hand_f_solution.values()) == hand_total_solution == oracle([2, 4, 8])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [40]:
def exercise_3_dp_solution(arr: List[int]) -> int:
    mod = 10 ** 9 + 7
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i, a in enumerate(arr):
        for j in range(i):
            b = arr[j]
            if a % b == 0 and (c := a // b) in idx:
                f[i] = (f[i] + f[j] * f[idx[c]]) % mod
    return sum(f) % mod

assert exercise_3_dp_solution([2, 4]) == 3
assert exercise_3_dp_solution([2, 4, 5, 10]) == 7
assert exercise_3_dp_solution([2, 4, 8]) == 8
rng = random.Random(3)
for _ in range(100):
    arr = rng.sample(list(range(2, 40)), rng.randint(1, 6))
    assert exercise_3_dp_solution(arr) == oracle(arr), arr
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [41]:
def exercise_4_push_solution(arr: List[int]) -> int:
    mod = 10 ** 9 + 7
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    f = [1] * n
    for i in range(n):
        b = arr[i]
        for j in range(i + 1):
            k = idx.get(arr[j] * b)
            if k is not None:
                f[k] = (f[k] + f[j] * f[i] * (1 if i == j else 2)) % mod
    return sum(f) % mod

assert exercise_4_push_solution([2, 4]) == 3
assert exercise_4_push_solution([2, 4, 5, 10]) == 7
assert exercise_4_push_solution([2, 3, 6]) == 5
rng = random.Random(4)
for _ in range(100):
    arr = rng.sample(list(range(2, 40)), rng.randint(1, 6))
    assert exercise_4_push_solution(arr) == oracle(arr), arr
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [42]:
def exercise_5_height_solution(arr: List[int], h: int) -> int:
    mod = 10 ** 9 + 7
    arr = sorted(arr)
    n = len(arr)
    idx = {v: i for i, v in enumerate(arr)}
    prev = [1] * n                                   # f_1: chỉ các nút đơn
    for _ in range(h - 1):
        cur = [1] * n
        for i, a in enumerate(arr):
            for j in range(i):
                b = arr[j]
                if a % b == 0 and (c := a // b) in idx:
                    cur[i] = (cur[i] + prev[j] * prev[idx[c]]) % mod
        prev = cur
    return sum(prev) % mod

def height(tree):
    return 1 if len(tree) == 1 else 1 + max(height(tree[1]), height(tree[2]))

def height_oracle(arr, h):
    return sum(1 for v in arr for t in enumerate_trees(arr, v) if height(t) <= h)

assert exercise_5_height_solution([2, 4, 5, 10], 1) == 4
assert exercise_5_height_solution([2, 4, 5, 10], 2) == 7
assert exercise_5_height_solution([2, 4, 8], 2) == 6
assert exercise_5_height_solution([2, 4, 8], 3) == 8
rng = random.Random(5)
for _ in range(100):
    arr = rng.sample([2, 3, 4, 6, 8, 9, 12, 16, 18, 24, 36], rng.randint(1, 6))
    for h in (1, 2, 3, 4, 6):
        assert exercise_5_height_solution(arr, h) == height_oracle(arr, h), (arr, h)
    assert exercise_5_height_solution(arr, 30) == Solution().numFactoredBinaryTrees(list(arr))
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DP sắp xếp, kéo)

```python
class Solution:
    def numFactoredBinaryTrees(self, arr: List[int]) -> int:
        mod = 10**9 + 7
        n = len(arr)
        arr.sort()                                         # nhỏ trước
        idx = {v: i for i, v in enumerate(arr)}            # giá trị -> chỉ số
        f = [1] * n                                        # f[i]: số cây gốc arr[i]; 1 = cây một nút
        for i, a in enumerate(arr):
            for j in range(i):
                b = arr[j]
                if a % b == 0 and (c := a // b) in idx:    # a = b * c
                    f[i] = (f[i] + f[j] * f[idx[c]]) % mod
        return sum(f) % mod
```

Công thức nhớ nhanh: `f(v) = 1 + Σ f(a) * f(b)` trên các cặp **có thứ tự** `(a, b)`, `a * b = v`; đáp án `Σ f(v) mod 1e9+7`.

### Checklist trước khi submit

- [ ] Đã sắp xếp `arr` tăng dần trước khi DP?
- [ ] `f = [1] * n` (mỗi giá trị là một cây một nút)?
- [ ] Kiểm tra `a % b == 0` **trước** khi lấy `a // b` (không dùng `/`)?
- [ ] Đếm cả hai thứ tự `(b, c)` và `(c, b)` khi `b != c`?
- [ ] `% mod` sau mỗi cập nhật và ở tổng cuối?
- [ ] Dùng `dict` giá trị -> chỉ số để tra `c` trong `O(1)`?
- [ ] Đã thử: một phần tử, không có quan hệ nhân, `[2, 3, 6]`, lũy thừa của 2 tới `2^29`?
- [ ] Tên method đúng `numFactoredBinaryTrees(self, arr)` và trả `int`?